In [ ]:
import pandas as pd
import numpy as np

# 假设文件是 CSV，编码通常是 GBK 或 UTF-8
df = pd.read_csv("ctp_tick_raw.csv", encoding="gbk")

# 强制将时间字段转为字符串，避免 Pandas 自动转换时出错
df["TradingDay"] = df["TradingDay"].astype(str)
df["UpdateTime"] = df["UpdateTime"].astype(str)

print(df.dtypes)
print(df.head())

def filter_trading_hours(df):
    """过滤非交易时段的冗余/测试数据"""
    df = df.copy()
    # 提取小时和分钟用于判断
    df["_hour"] = pd.to_datetime(df["UpdateTime"], format="%H:%M:%S").dt.hour
    df["_min"] = pd.to_datetime(df["UpdateTime"], format="%H:%M:%S").dt.minute

    # 定义需要过滤的“垃圾时段”：
    # 15:30 ~ 20:40（下午收盘后到夜盘前）
    # 03:00 ~ 08:40（夜盘结束后到早盘前）
    mask_bad_1 = (df["_hour"] == 15) & (df["_min"] >= 30) | (df["_hour"] > 15) & (df["_hour"] < 20)
    mask_bad_2 = (df["_hour"] == 20) & (df["_min"] < 40)
    mask_bad_3 = (df["_hour"] < 3) & (df["_hour"] >= 0)  # 凌晨0-3点，视品种而定
    mask_bad_4 = (df["_hour"] == 3) & (df["_min"] == 0)
    mask_bad_5 = (df["_hour"] > 3) & (df["_hour"] < 9)  # 3点到9点之间

    df = df[~(mask_bad_1 | mask_bad_2 | mask_bad_5)]
    df = df.drop(columns=["_hour", "_min"])
    return df

def fix_trading_day(df):
    """
    修正交易日逻辑：
    - TradingDay 应为夜盘所属的“交易日”
    - ActionDay 为实际自然日
    - 夜盘数据（20:40之后）的 TradingDay 应设为次日
    """
    df = df.copy()

    # 确保 TradingDay 是日期格式
    df["TradingDay"] = pd.to_datetime(df["TradingDay"], format="%Y%m%d")

    # 识别夜盘时段：20:40 之后（含零点后凌晨）
    time_dt = pd.to_datetime(df["UpdateTime"], format="%H:%M:%S")
    is_night = (time_dt.dt.hour >= 20) | (time_dt.dt.hour < 4)

    # 对于夜盘数据，如果 TradingDay 等于 ActionDay（当天），说明日期未修正
    # 需要将 TradingDay 调整为下一个交易日
    # 注意：这里需要交易日历，简化处理：夜盘数据的 TradingDay 应 > ActionDay
    df.loc[is_night, "TradingDay"] = df.loc[is_night].apply(
        lambda r: r["TradingDay"] if r["TradingDay"] > pd.to_datetime(r["ActionDay"])
        else r["TradingDay"] + pd.Timedelta(days=1),
        axis=1
    )
    return df

def deduplicate_and_fix_timestamp(df):
    """去重并处理时间戳冲突"""
    df = df.copy()

    # 构造完整时间戳
    df["datetime"] = pd.to_datetime(
        df["ActionDay"].astype(str) + " " + df["UpdateTime"],
        format="%Y%m%d %H:%M:%S"
    )

    # 对于郑商所品种，对同一秒内的多条 tick，人为递增毫秒
    # 先检测重复
    dup_mask = df.duplicated(subset=["InstrumentID", "datetime"], keep=False)

    if dup_mask.any():
        # 对重复组内的记录按顺序赋毫秒值
        df.loc[dup_mask, "UpdateMillisec"] = df.loc[dup_mask].groupby(
            ["InstrumentID", "datetime"]
        ).cumcount() * 500  # 假设 500ms 间隔

        # 重建毫秒级时间戳
        df["datetime"] = df["datetime"] + pd.to_timedelta(
            df["UpdateMillisec"], unit="ms"
        )

    # 全局去重
    df = df.drop_duplicates(subset=["InstrumentID", "datetime"], keep="first")
    return df

def clean_price_anomalies(df, price_columns=None):
    """清洗价格字段中的异常浮点数值"""
    df = df.copy()

    if price_columns is None:
        price_columns = [c for c in df.columns if "Price" in c]

    for col in price_columns:
        # 将异常大的浮点数替换为 NaN
        df[col] = pd.to_numeric(df[col], errors="coerce")

        # 过滤掉超过合理范围的值（期货价格通常不会超过 1e6）
        df.loc[df[col] > 1e6, col] = np.nan
        df.loc[df[col] <= 0, col] = np.nan

    return df

def fill_and_cast(df):
    """填充缺失值并统一数据类型"""
    df = df.copy()

    # 价格类字段：前向填充（非连续缺失）
    price_cols = [c for c in df.columns if "Price" in c]
    df[price_cols] = df[price_cols].ffill()

    # 数量类字段：0 或前向填充
    vol_cols = ["Volume", "OpenInterest"]
    for c in vol_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0).astype("int64")

    # 合约代码统一大写
    if "InstrumentID" in df.columns:
        df["InstrumentID"] = df["InstrumentID"].str.upper()

    return df

def clean_ctp_data(filepath):
    df = pd.read_csv(filepath, encoding="gbk")
    df = filter_trading_hours(df)
    df = fix_trading_day(df)
    df = deduplicate_and_fix_timestamp(df)
    df = clean_price_anomalies(df)
    df = fill_and_cast(df)
    return df

if __name__ == "__main__":
    cleaned = clean_ctp_data("ctp_tick_raw.csv")
    cleaned.to_parquet("ctp_tick_cleaned.parquet", index=False)
    print(f"清洗完成，共 {len(cleaned)} 条记录")